# Competencia: ¿qué está contratando el Estado?

**Sesión 6 · IA Intermedio · EstudIA 2026**

Cada texto es el objeto de un contrato público firmado en Colombia en 2025, tal como lo escribió la entidad en SECOP
II: “suministro de combustible para el parque automotor…”, “prestación de servicios profesionales como médico
general…”. Tu modelo tiene que decir la **categoría** de lo que se contrató, entre **15**: salud, construcción,
educación, combustibles, papelería, vigilancia y seguridad…

**Solo hay 32 contratos etiquetados por categoría.** Es la situación de muchos proyectos reales, y es donde los
embeddings brillan.

**La métrica: exactitud.** La prueba tiene 200 contratos de cada categoría, así que la exactitud y la exactitud
balanceada de la sesión 4 dan lo mismo. Responder siempre la misma categoría da 6,7 %.

**La línea que hay que superar: una bolsa de palabras** entrenada con los mismos 480 contratos.

**Reglas**

1. Tu modelo convierte cada texto en un vector con un **modelo de embeddings de Hugging Face, congelado**, y encima
   pone un **clasificador pequeño** (regresión logística, vecinos más cercanos o una red pequeña). También vale el
   zero-shot con embeddings. Nada de ajuste fino del transformer (llega en la sesión 7) ni de modelos que generan
   texto.
2. Tienes 480 contratos con su categoría y 3.000 sin categoría. No tienes las respuestas de prueba: valida con
   **validación cruzada** sobre tus 480.
3. Envía tus predicciones todas las veces que quieras (una vez cada 10 segundos). Gana la mayor **exactitud**.
4. La tabla en vivo usa **la mitad** de los contratos de prueba. Al final, el profesor revela el resultado con **la
   otra mitad**: ese decide la competencia.

**Cómo empezar.** *Archivo › Guardar una copia en Drive*. Con CPU funciona; para probar modelos grandes, activa una
GPU (*Entorno de ejecución › Cambiar tipo de entorno de ejecución › GPU T4*). Ejecuta todo el notebook (*Entorno de
ejecución › Ejecutar todas*). Cuando te pida la clave, cópiala de la página de la competencia en la plataforma y
pégala. Así haces tu primer envío. Después, mejora tu modelo.

In [ ]:
# Preparación: librerías y datos. Ejecútala una vez.
import os, re, time, urllib.request
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s06")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

try:
    from sentence_transformers import SentenceTransformer
except ImportError:  # Colab ya la trae; fuera de Colab puede faltar
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    from sentence_transformers import SentenceTransformer

def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

datos = pd.read_csv(descargar("contratos_entrenamiento.csv"))
prueba = pd.read_csv(descargar("contratos_prueba.csv"))
categorias = pd.read_csv(descargar("categorias.csv"))
CLASES = sorted(datos.categoria.unique())
GPU = torch.cuda.is_available()
print(f"Listo: {miles(len(datos))} contratos con categoría, {miles(len(prueba))} de prueba, {len(CLASES)} categorías.",
      "Con GPU." if GPU else "Sin GPU: funciona con los modelos pequeños.")

In [ ]:
# @title Función para enviar tus predicciones (no hace falta cambiarla)
import getpass, json, urllib.error

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")

def enviar(ids, predicciones, nota=""):
    """Envía tus predicciones a la plataforma del curso y muestra tu exactitud en la tabla.

    ids: los ids de los contratos de prueba. predicciones: una categoría por contrato, como "salud".
    nota: un texto corto para recordar qué probaste (opcional)."""
    global _CLAVE
    if not _CLAVE:
        _CLAVE = getpass.getpass("Pega tu clave (plataforma › Competencia › Copiar) y oprime Enter: ").strip()
    cuerpo = json.dumps({"ids": [int(i) for i in ids], "predicciones": [str(p) for p in predicciones],
                         "nota": str(nota)[:80]}).encode()
    req = urllib.request.Request(f"{PLATAFORMA}/api/competencia/s06/envios", data=cuerpo, method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {_CLAVE}",
                                          "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""  # la próxima vez la pide de nuevo
        print("No se pudo enviar:", res.get("error", f"error {e.code}"))
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    metrica = res.get("metrica", "Exactitud")
    print(f"Envío {res['envio']} recibido. {metrica} en la tabla: {pct(res.get('puntaje', res['exactitud']))}.")
    print(f"Tu mejor resultado: {pct(res['mejor'])} · puesto {res['puesto']} de {res['participantes']} ({res['alias']}).")

---
## 1. Los datos

In [ ]:
for _, fila in datos.sample(8, random_state=3).iterrows():
    print(f"{fila.categoria:15s} {fila.texto[:105]}")
print("\nContratos por categoría:", datos.categoria.value_counts().unique().tolist(),
      "· palabras por texto (mediana):", int(datos.texto.str.split().str.len().median()))
print("\nLas categorías:")
print(categorias.to_string(index=False))

Las categorías las eligió cada entidad al publicar su contrato, así que algunas etiquetas son discutibles. Quitamos
los errores evidentes, pero queda algo de ruido, como en cualquier proyecto real.

### 1.1 Cómo validar con 480 ejemplos

Si apartas el 20 % para validar, entrenas con menos ejemplos y mides con muy pocos. Con **validación cruzada** partimos
los 480 en 5 grupos; cada grupo se predice con un modelo entrenado con los otros 4. Así cada contrato se predice sin
haberlo visto y medimos con los 480.

### 1.2 La línea que hay que superar: una bolsa de palabras

In [ ]:
CV = StratifiedKFold(5, shuffle=True, random_state=0)

def validacion_cruzada(X, clasificador):
    """Exactitud de validación cruzada (5 grupos) de un clasificador sobre la matriz X de los 480 contratos."""
    pred = cross_val_predict(clasificador, X, datos.categoria.to_numpy(), cv=CV)
    return (pred == datos.categoria.to_numpy()).mean()

bolsa = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2))
LINEA_BASE = validacion_cruzada(bolsa.fit_transform(datos.texto), LogisticRegression(max_iter=3000, C=10))
print("Bolsa de palabras · exactitud de validación cruzada:", pct(LINEA_BASE))

---
## 2. Tu modelo

Las piezas que puedes cambiar:

- `MODELO`: el modelo de embeddings, cualquiera de Hugging Face que funcione con `sentence-transformers`.
- `PREFIJO`: el texto que va antes de cada contrato. Los modelos e5 piden `"query: "`; otros no piden nada. Lo dice
  la ficha de cada modelo.
- `MINUSCULAS`: si es `True`, pasamos los textos a minúsculas antes de convertirlos.
- `CLASIFICADOR`: `"regresion"` (regresión logística) o `"vecinos"` (los *k* vecinos más cercanos).
- `C`: para la regresión logística; más grande, más se ajusta a los ejemplos. `VECINOS`: para los vecinos.

Así como vienen funcionan, pero dejan mucho por ganar. Ejecútalas tal cual para tu primer envío y después mejóralas.

In [ ]:
MODELO = "intfloat/multilingual-e5-small"
PREFIJO = "query: "
MINUSCULAS = False
CLASIFICADOR = "regresion"
C = 1.0
VECINOS = 10

In [ ]:
_modelos, _vectores = {}, {}

def embeber(textos, modelo=None, prefijo=None, minusculas=None):
    """Los embeddings de una lista de textos. Guarda lo que ya calculó: cambiar C no vuelve a pasar los textos."""
    modelo = modelo or MODELO
    prefijo = PREFIJO if prefijo is None else prefijo
    minusculas = MINUSCULAS if minusculas is None else minusculas
    clave = (modelo, prefijo, minusculas, len(textos), hash(tuple(textos)))
    if clave not in _vectores:
        if modelo not in _modelos:
            _modelos[modelo] = SentenceTransformer(modelo, device="cuda" if GPU else "cpu", trust_remote_code=True)
        listos = [prefijo + (t.lower() if minusculas else t) for t in textos]
        inicio = time.time()
        _vectores[clave] = _modelos[modelo].encode(listos, batch_size=32, normalize_embeddings=True,
                                                   show_progress_bar=False)
        print(f"{miles(len(textos))} textos convertidos con {modelo} en {time.time() - inicio:.0f} s.")
    return _vectores[clave]

def crear_clasificador():
    if CLASIFICADOR == "vecinos":
        return KNeighborsClassifier(n_neighbors=VECINOS, metric="cosine", weights="distance")
    return LogisticRegression(max_iter=5000, C=C)

### 2.1 Validar

In [ ]:
X = embeber(list(datos.texto))
EXACTITUD_VALIDACION = validacion_cruzada(X, crear_clasificador())
print(f"Tu modelo · exactitud de validación cruzada: {pct(EXACTITUD_VALIDACION)} · bolsa de palabras: {pct(LINEA_BASE)}")

### 2.2 Entrenar con los 480, predecir la prueba y enviar

Una vez validado, el clasificador se entrena con los 480 contratos. Cambia la nota para recordar qué probaste en cada
envío. La primera vez te pide la clave.

In [ ]:
clasificador = crear_clasificador().fit(X, datos.categoria)
predicciones = clasificador.predict(embeber(list(prueba.texto)))
enviar(prueba.id, predicciones, nota="primer envío")

---
## 3. Pistas para subir en la tabla

Prueba una cosa a la vez, mira la exactitud de **validación cruzada** y envía cuando mejore.

<details><summary>Pista 1</summary>

Mira los textos de la sección 1: ¿cómo están escritos? Los modelos distinguen mayúsculas de minúsculas, y casi todo lo
que leyeron estaba escrito en minúsculas.
</details>

<details><summary>Pista 2</summary>

`C = 1` es prudente. Con 32 ejemplos por categoría, prueba valores más grandes (10, 100) y mira la validación.
</details>

<details><summary>Pista 3</summary>

Un modelo más grande. Algunos que funcionan con español: `"intfloat/multilingual-e5-base"` (278 millones de
parámetros), `"intfloat/multilingual-e5-large"` (560 millones), `"BAAI/bge-m3"` (568 millones, sin prefijo) y
`"Qwen/Qwen3-Embedding-0.6B"` (600 millones, sin prefijo). Los grandes necesitan GPU. Cambia `PREFIJO` según la ficha
del modelo.
</details>

<details><summary>Pista 4</summary>

Junta dos modelos: convierte los textos con cada uno y pega los dos vectores, uno al lado del otro.

```python
X = np.hstack([embeber(list(datos.texto), modelo="intfloat/multilingual-e5-base"),
               embeber(list(datos.texto), modelo="BAAI/bge-m3", prefijo="")])
```

Recuerda hacer lo mismo con la prueba.
</details>

<details><summary>Pista 5</summary>

Mira qué categorías confunde tu modelo: con las predicciones de la validación cruzada puedes armar una matriz de
confusión. ¿Las confusiones tienen sentido? ¿Algunas etiquetas son discutibles?
</details>

**Una advertencia.** Si eliges tus cambios mirando solo la tabla en vivo, terminas ajustando el modelo a esa mitad
de la prueba. El resultado final se calcula con la otra mitad.

**Fuente de los datos:** SECOP II – Contratos Electrónicos, Agencia Nacional de Contratación Pública – Colombia
Compra Eficiente, publicado en datos.gov.co. Licencia CC BY-SA 4.0. Usamos el objeto de contratos firmados en 2025;
la categoría es el segmento del código UNSPSC principal que eligió la entidad. Quitamos números de proceso, textos que
parecían nombrar a una persona, duplicados y las etiquetas evidentemente equivocadas.